<a href="https://colab.research.google.com/github/saisathwik2703/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saisathwik2703/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [17]:
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

print("Token loaded successfully")

Token loaded successfully


In [18]:
!pip -q install duckdb pandas numpy scikit-learn

In [19]:
import os
import duckdb
import pandas as pd
import numpy as np

# Get HF token from Colab Secrets
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN secret was not found. Add it in Colab Secrets.")

# Connect DuckDB
con = duckdb.connect()

# Keep token out of SQL text
con.execute("SET VARIABLE hf_token = ?", [HF_TOKEN])
con.execute("""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN getvariable('hf_token')
    )
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

FACT = f"{REL}/fact_content_daily_performance/month=2026-03/*.parquet"
CONTENT = f"{REL}/dim_content.parquet"

print("DuckDB connected successfully.")
print("Feature/verification window: March 2026")

DuckDB connected successfully.
Feature/verification window: March 2026


In [20]:
con.sql(f"""
    SELECT *
    FROM read_parquet('{FACT}')
    LIMIT 5
""").df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** One row represents one content page for one client on one report date in `fact_content_daily_performance`.

**Time window:** March 2026 (`month=2026-03`), a mid-panel month. I do not use the June 2026 `_sample` month for development because it is the final outcome month.

**Prediction/ranking target:** Whether a content page shows a directional decline in search performance. The future outcome is used only as a label/proxy and is not included in the feature inputs.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# QUERY 1 — Verify the grain

con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        report_date,
        COUNT(*) AS n
    FROM read_parquet('{FACT}')
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").show()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────┬────────────────┬─────────────┬───────┐
│ content_hash_id │ client_hash_id │ report_date │   n   │
│     varchar     │    varchar     │    date     │ int64 │
├─────────────────┴────────────────┴─────────────┴───────┤
│                         0 rows                         │
└────────────────────────────────────────────────────────┘



## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features

1. `gsc_impressions` — search visibility measured through impressions.
2. `gsc_clicks` — search clicks observed during the feature window.
3. `gsc_avg_position` — average search position observed during the feature window.
4. `content_created_date` — used to calculate content age at the decision date.
5. `ga4_data_available` — indicates whether GA4 data was available for the observation.

### Label / proxy

A future-window decline indicator based on search impressions. This is used to evaluate the outcome and demonstrate leakage, but it is not included as an honest feature.

### Context

`content_hash_id` and `client_hash_id` are used for grouping, joins and validation. They are not treated as predictive features.

### Excluded

`trend_direction` and `trend_pct` are excluded because they are derived from performance movement and can directly encode the outcome.

Raw URLs, query text, client names and other identifying information are excluded because the assignment requires pseudonymized data and no client-identifying output.

In [22]:
# This cell is for CODE (numbers, a query, a check).
# QUERY 2 — Row count and date span

con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM read_parquet('{FACT}')
""").show()


┌───────────┬────────────┬────────────┐
│ row_count │  min_date  │  max_date  │
│   int64   │    date    │    date    │
├───────────┼────────────┼────────────┤
│   9841378 │ 2026-03-01 │ 2026-03-31 │
└───────────┴────────────┴────────────┘



## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [23]:
# This cell is for CODE (numbers, a query, a check).
# QUERY 3 — Availability using IS TRUE

con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(
            CASE
                WHEN ga4_data_available IS TRUE THEN 1
                ELSE 0
            END
        ) AS ga4_available_rows
    FROM read_parquet('{FACT}')
""").show()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────┐
│ total_rows │ ga4_available_rows │
│   int64    │       int128       │
├────────────┼────────────────────┤
│    9841378 │             413966 │
└────────────┴────────────────────┘



In [24]:
# STEP 4 — Build the five-feature frame

features = con.sql(f"""
    SELECT
        f.content_hash_id,

        SUM(f.gsc_impressions) AS impressions_march,

        SUM(f.gsc_clicks) AS clicks_march,

        AVG(f.gsc_avg_position) AS avg_position_march,

        DATEDIFF(
            'day',
            c.content_created_date,
            DATE '2026-03-31'
        ) AS content_age_days,

        SUM(
            CASE
                WHEN f.ga4_data_available IS TRUE THEN 1
                ELSE 0
            END
        ) AS days_with_ga4

    FROM read_parquet('{FACT}') f

    JOIN read_parquet('{CONTENT}') c
        ON f.content_hash_id = c.content_hash_id

    GROUP BY
        f.content_hash_id,
        c.content_created_date
""").df()

print("Feature frame shape:", features.shape)

features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (331437, 6)


,content_hash_id,impressions_march,clicks_march,avg_position_march,content_age_days,days_with_ga4
0,content_b7e512995f79d5a6,1140.0,2.0,4.394234,396,0.0
1,content_a7da352b73b02668,4944.0,13.0,7.244844,396,2.0
2,content_d056587ff7faca0c,2770.0,16.0,4.459107,396,2.0
3,content_bfd1e41c2af250c8,48.0,0.0,14.753175,396,0.0
4,content_2662845f598544ef,150.0,1.0,6.341880,396,1.0


### Why each feature is available at decision time

1. `impressions_march` — knowable because March search impressions have already been observed by the decision date.

2. `clicks_march` — knowable because March search clicks are already available at the decision date.

3. `avg_position_march` — knowable because search position is observed during the March feature window.

4. `content_age_days` — knowable because the content creation date is historical information.

5. `days_with_ga4` — knowable because it counts GA4-available days observed within the March window.

In [25]:
# STEP 5 — Deliberate leakage experiment

APRIL = f"{REL}/fact_content_daily_performance/month=2026-04/*.parquet"

# Future April information
april = con.sql(f"""
    SELECT
        content_hash_id,
        SUM(gsc_impressions) AS impressions_april
    FROM read_parquet('{APRIL}')
    GROUP BY content_hash_id
""").df()

df = features.merge(
    april,
    on="content_hash_id",
    how="left"
)

# Future outcome / label
df["is_declining"] = (
    df["impressions_april"] < df["impressions_march"]
).astype(int)

# INTENTIONAL LEAK:
# April information is being used to predict an April outcome.
df["leaky_signal"] = df["impressions_april"]

print("Rows:", len(df))
print("Columns:", df.columns.tolist())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 331437
Columns: ['content_hash_id', 'impressions_march', 'clicks_march', 'avg_position_march', 'content_age_days', 'days_with_ga4', 'impressions_april', 'is_declining', 'leaky_signal']


In [26]:
# STEP 5B — Measure the deliberately leaky model

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

X_leaky = df[
    [
        "impressions_march",
        "avg_position_march",
        "leaky_signal"
    ]
].fillna(0)

y = df["is_declining"]

model_leaky = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model_leaky.fit(X_leaky, y)

auc_leaky = roc_auc_score(
    y,
    model_leaky.predict_proba(X_leaky)[:, 1]
)

print("AUC WITH DELIBERATE LEAKAGE:", auc_leaky)

AUC WITH DELIBERATE LEAKAGE: 0.9999997873309947


### Leakage lesson

The leaky model uses April impressions while predicting an April outcome. April information would not be available at the March decision moment. The resulting score is therefore not an honest estimate of predictive performance.

The future-derived `leaky_signal` must be removed from the final feature set.

In [27]:
# STEP 6 — Remove leakage and calculate the honest score

honest_features = [
    "impressions_march",
    "clicks_march",
    "avg_position_march",
    "content_age_days",
    "days_with_ga4"
]

X_honest = df[honest_features].fillna(0)
y = df["is_declining"]

model_honest = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model_honest.fit(X_honest, y)

auc_honest = roc_auc_score(
    y,
    model_honest.predict_proba(X_honest)[:, 1]
)

print("AUC WITHOUT LEAKAGE:", auc_honest)

AUC WITHOUT LEAKAGE: 0.8265404739318363


In [28]:
# Confirm that the leakage column is not in the final features

print("Final features:")
for feature in honest_features:
    print("-", feature)

print("\nLeakage removed:", "leaky_signal" not in honest_features)

Final features:
- impressions_march
- clicks_march
- avg_position_march
- content_age_days
- days_with_ga4

Leakage removed: True


### Final leakage check

The deliberately leaked future feature was removed from the final feature set. The final model uses only information available from the March decision window. The honest AUC is reported from the model without the leakage column.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data limits

- The March slice is an unbalanced history, so different clients/content items may have different amounts of historical data.
- Some early rows may have GSC data without GA4 availability, so GA4-based information is not available uniformly.
- A 90-day historical window can overlap with other analysis windows, so future information must not be used as a feature.
- This analysis uses March 2026 as one development month, so the observed patterns may not represent every month.

In [29]:
# This cell is for CODE (numbers, a query, a check).
# STEP 7 — Final feature-frame check

print("Final feature frame shape:", features.shape)

print("\nFinal features:")
print(features.columns.tolist())

print("\nMissing values:")
print(features.isna().sum())


Final feature frame shape: (331437, 6)

Final features:
['content_hash_id', 'impressions_march', 'clicks_march', 'avg_position_march', 'content_age_days', 'days_with_ga4']

Missing values:
content_hash_id            0
impressions_march          0
clicks_march               0
avg_position_march    154699
content_age_days           0
days_with_ga4              0
dtype: int64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.